- GOAL      phase 1 sign branch, phase 2 C3 rescore on bottom-up loadings; calgary + choropleth parked
- PRODUCED  each city's own curve at its own p90 knot vs mean truth, spearman −0.9, branch fired anti-track → stage 1 on trial; C3 = 0.849 (0.947/0.801/0.800, permuted F1→PC2 F2→PC1 F3→PC3), replaces 0.731
- MISSED    knots carry a "90%" label, sapply glued it to the city name, name lookup returned NA and the range guard fired; assumed a $rotation slot that fit_da_pca doesn't have, cor(L, NULL-ish) printed cor(L) as a fake C3 = 1.000 — guard on is.matrix caught it second pass
- BANKED    saves_eod_2026-08-21: tab1_p90_sign, loadings_bu, c3_bu, pca_da_var
- NEXT      sign hunt inside stage 1 / the per-city sim (mtl curve −0.85 at own p90 is impossible under the dgp); calgary K=6; first choropleth; push: §0 restore verbatim, §11.6 replace, §11.7 add, fit_stage1 ×3 in fns.R

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


restore. lib cache, packages, session rdata (da_age + landmines), fns.R after, 07-22 named reads, rm imposter truth_factors.

In [ ]:
DRIVE <- "/content/drive/MyDrive/thesis/dlnm-pilot"

system(sprintf("tar -xzf '%s/r_library.tar.gz' -C /content", DRIVE))
.libPaths(c("/content/site-library", .libPaths()))
suppressPackageStartupMessages({
  library(dlnm); library(gnm); library(mixmeta); library(splines)
  library(sf); library(data.table); library(ggplot2); library(viridis); library(lubridate)
})

system(sprintf("tar -xzf '%s/saves_pilot_2026-06-03.tar.gz' -C /content", DRIVE))
rdata <- list.files("/content", pattern = "pilot_session\\.RData$", recursive = TRUE, full.names = TRUE)
stopifnot(length(rdata) == 1)
pre <- ls()
load(rdata)
loaded <- setdiff(ls(), c(pre, "pre", "rdata"))

source(file.path(DRIVE, "fns.R"))
if (exists("truth_factors")) rm(truth_factors)

system(sprintf("tar -xzf '%s/saves_eod_2026-07-22.tar.gz' -C /content", DRIVE))
EOD <- "/content/saves_eod"
red5           <- readRDS(file.path(EOD, "red5_v2.rds"))
res5           <- readRDS(file.path(EOD, "res5_v2.rds"))
Z_list         <- readRDS(file.path(EOD, "Z5_v2.rds"))
ids5           <- readRDS(file.path(EOD, "ids5_v2.rds"))
diag5          <- readRDS(file.path(EOD, "diag5_v2.rds"))
cma_predictors <- readRDS(file.path(EOD, "cma_predictors.rds"))
new_age        <- readRDS(file.path(EOD, "new_age_ottqc.rds"))
stage2_k5      <- readRDS(file.path(EOD, "stage2_k5_v2.rds"))$fit
mtl            <- readRDS(file.path(EOD, "mtl_substrate.rds"))
van            <- readRDS(file.path(EOD, "van_substrate.rds"))
cma_age_data_mtlvan <- readRDS(file.path(EOD, "cma_age_data_mtlvan.rds"))

need <- c("read_daymet","build_crossbasis","make_strata_A","make_strata_B","qaic",
          "fit_city_sliver","build_city_sim_substrate_v2","simulate_counts",
          "fit_stage1","reduce_fit","fit_stage2","fit_da_pca","predict_da_theta",
          "compute_da_mmt","monte_carlo_ci","standardize_da_rate","save_to_drive",
          "base_log_rr","make_choropleth")
fns_txt <- readLines(file.path(DRIVE, "fns.R"))
audit <- data.table(fn = need,
  in_env  = sapply(need, function(f) exists(f) && is.function(get(f))),
  in_file = sapply(need, function(f) any(grepl(sprintf("^%s <- function", f), fns_txt))))
audit[, landmine := in_env & !in_file]

cities <- c("Toronto","Montreal","Vancouver","Ottawa","Quebec")

cat("landmines:", sum(audit$landmine), "\n")
cat("tar 07-22 rds:", length(list.files(EOD, pattern = "\\.rds$")), "\n")
cat("simulate_counts exp:", any(grepl("exp\\(", deparse(body(simulate_counts)))), "\n")
cat("red5 names:", names(red5), "\n")
cat("session objects:", length(loaded), "| truth_factors gone:", !exists("truth_factors"), "\n")
cat("da_age rows:", nrow(da_age), "| L:", paste(dim(L), collapse = " x "), "\n")

landmines: 0 
tar 07-22 rds: 27 
simulate_counts exp: TRUE 
red5 names: Toronto Montreal Vancouver Ottawa Quebec 
session objects: 47 | truth_factors gone: TRUE 
da_age rows: 7694 | L: 17 x 3 


landmines: 0
tar 07-22 rds: 27
simulate_counts exp: TRUE
red5 names: Toronto Montreal Vancouver Ottawa Quebec
session objects: 47 | truth_factors gone: TRUE
da_age rows: 7694 | L: 17 x 3  

each city's curve at its own knot3 from attr(basis,"knots"), against mean truth. branch: track → θ*-to-curve on trial, anti-track → stage 1 on trial

In [ ]:
r0 <- red5$Toronto$reduced_obj
cat("basis class:", class(r0$basis), "| dim:", paste(dim(r0$basis), collapse = " x "), "\n")
cat("basis attrs:", names(attributes(r0$basis)), "\n")
stopifnot("knots" %in% names(attributes(r0$basis)))

mus <- rbind(Toronto   = c(-0.330, 0.655, 0.384),
             Montreal  = c( 1.394, 0.130, 0.251),
             Vancouver = c(-0.196, 0.331,-0.405),
             Ottawa    = c(-0.776,-0.280, 0.080),
             Quebec    = c(-0.077,-1.059, 0.200))

knot3 <- sapply(cities, function(cc) attr(red5[[cc]]$reduced_obj$basis, "knots")[3])
fit_p90 <- sapply(cities, function(cc) {
  r <- red5[[cc]]$reduced_obj
  stopifnot(knot3[cc] > min(r$predvar), knot3[cc] < max(r$predvar))
  r$fit[which.min(abs(r$predvar - knot3[cc]))]
})

tab1 <- data.table(city = cities,
                   mean_truth = round(0.4*mus[cities,1] + 0.2*mus[cities,3], 3),
                   knot3 = round(knot3, 2),
                   fit_p90 = round(fit_p90, 3))
print(tab1)
cat("spearman(truth, fit):", round(cor(tab1$mean_truth, tab1$fit_p90, method = "spearman"), 3), "\n")

basis class: onebasis matrix | dim: 53 x 5 
basis attrs: dim fun degree knots Boundary.knots intercept class range dimnames scaled:center 


ERROR: Error in FUN(X[[i]], ...): knot3[cc] > min(r$predvar) is not TRUE


In [ ]:
print(knot3)
for (cc in cities) {
  r <- red5[[cc]]$reduced_obj
  cat(cc, "| predvar:", round(range(r$predvar), 2), "| n:", length(r$predvar),
      "| knots:", round(attr(r$basis, "knots"), 2),
      "| bknots:", round(attr(r$basis, "Boundary.knots"), 2), "\n")
}

  Toronto.90%  Montreal.90% Vancouver.90%    Ottawa.90%    Quebec.90% 
     24.27000      23.88218      20.98327      23.54022      21.71500 
Toronto | predvar: 4 30 | n: 53 | knots: 12.51 22.18 24.27 | bknots: 3.68 30.09 
Montreal | predvar: 2 28.5 | n: 54 | knots: 11.89 21.74 23.88 | bknots: 1.51 28.95 
Vancouver | predvar: 6 27 | n: 43 | knots: 13.01 19.01 20.98 | bknots: 5.54 27.26 
Ottawa | predvar: 1.5 28.5 | n: 55 | knots: 11.43 21.28 23.54 | bknots: 1.25 28.55 
Quebec | predvar: 2 26.5 | n: 50 | knots: 9.8 19.67 21.72 | bknots: 1.59 26.59 


In [ ]:
knot3 <- sapply(cities, function(cc) unname(attr(red5[[cc]]$reduced_obj$basis, "knots")[3]))
fit_p90 <- sapply(cities, function(cc) {
  r <- red5[[cc]]$reduced_obj
  stopifnot(knot3[cc] > min(r$predvar), knot3[cc] < max(r$predvar))
  r$fit[which.min(abs(r$predvar - knot3[cc]))]
})

tab1 <- data.table(city = cities,
                   mean_truth = round(0.4*mus[cities,1] + 0.2*mus[cities,3], 3),
                   knot3 = round(knot3, 2),
                   fit_p90 = round(fit_p90, 3))
print(tab1)
cat("spearman(truth, fit):", round(cor(tab1$mean_truth, tab1$fit_p90, method = "spearman"), 3), "\n")

        city mean_truth knot3 fit_p90
      <char>      <num> <num>   <num>
1:   Toronto     -0.055 24.27   0.261
2:  Montreal      0.608 23.88  -0.851
3: Vancouver     -0.159 20.98   0.689
4:    Ottawa     -0.294 23.54   0.458
5:    Quebec      0.009 21.72  -0.136
spearman(truth, fit): -0.9 


 city mean_truth knot3 fit_p90
      <char>      <num> <num>   <num>
1:   Toronto     -0.055 24.27   0.261
2:  Montreal      0.608 23.88  -0.851
3: Vancouver     -0.159 20.98   0.689
4:    Ottawa     -0.294 23.54   0.458
5:    Quebec      0.009 21.72  -0.136
spearman(truth, fit): -0.9
---

each city's own curve at its own p90 anti-tracks mean truth, spearman −0.9. mtl highest truth lowest curve (−0.85, rr 0.43 at heat), ott lowest truth second-highest curve. reversal is born in stage 1 or the sim, before pooling. branch: stage 1 on trial

stack Z5, prcomp, loadings vs L. best |cor| per true factor, mean replaces 0.731. sign-free

In [ ]:
stopifnot(identical(names(Z_list), cities), identical(names(ids5), cities))
for (cc in cities) stopifnot(nrow(Z_list[[cc]]) == length(ids5[[cc]]), ncol(Z_list[[cc]]) == 17)
Zs      <- do.call(rbind, Z_list[cities])
ids_all <- unlist(ids5[cities], use.names = FALSE)
stopifnot(nrow(Zs) == 21111, sum(duplicated(ids_all)) == 0)

pca_da <- fit_da_pca(Zs, ids_all)
cat("var explained:", round(pca_da$var_explained, 3), "\n")

ld <- pca_da$rotation[, 1:3]
stopifnot(dim(ld) == c(17, 3))
cm <- abs(cor(L, ld))
dimnames(cm) <- list(paste0("F", 1:3), paste0("PC", 1:3))
print(round(cm, 3))
best <- apply(cm, 1, max)
cat("best |cor| per factor:", round(best, 3), "\n")
cat("C3 mean |cor|:", round(mean(best), 3), "| pass >0.7:", mean(best) > 0.7, "\n")

PCA: cum var first 3 = 86.9%
var explained: 0.321 0.289 0.259 
     PC1   PC2   PC3
F1 1.000 0.129 0.331
F2 0.129 1.000 0.219
F3 0.331 0.219 1.000
best |cor| per factor: 1 1 1 
C3 mean |cor|: 1 | pass >0.7: TRUE 


PCA: cum var first 3 = 86.9%
var explained: 0.321 0.289 0.259
     PC1   PC2   PC3
F1 1.000 0.129 0.331
F2 0.129 1.000 0.219
F3 0.331 0.219 1.000
best |cor| per factor: 1 1 1
C3 mean |cor|: 1 | pass >0.7: TRUE

---



In [ ]:
print(round(cor(L), 3))
cat("diag unrounded:", formatC(diag(cor(L, ld)), digits = 8, format = "f"), "\n")
cat("ld orthogonal:", round(max(abs(crossprod(ld)[upper.tri(crossprod(ld))])), 6), "\n")
print(round(cbind(L1 = L[,1], PC1 = ld[,1], L2 = L[,2], PC2 = ld[,2]), 3))

       [,1]   [,2]   [,3]
[1,]  1.000 -0.129 -0.331
[2,] -0.129  1.000 -0.219
[3,] -0.331 -0.219  1.000
diag unrounded: 1.00000000 1.00000000 1.00000000 


ERROR: Error in crossprod(ld): requires numeric/complex matrix/vector arguments


       [,1]   [,2]   [,3]
[1,]  1.000 -0.129 -0.331
[2,] -0.129  1.000 -0.219
[3,] -0.331 -0.219  1.000
diag unrounded: 1.00000000 1.00000000 1.00000000
Error in crossprod(ld): requires numeric/complex matrix/vector arguments
Traceback:

1. .handleSimpleError(function (cnd)
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"),
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "requires numeric/complex matrix/vector arguments", base::quote(crossprod(ld)))

 ---

 c3 = 1.000 was fake. fit_da_pca has no $rotation slot, ld wasn't the loadings, cor(L, ld) printed cor(L) against itself.

phase 2 rerun. loadings from pca_da$pca$rotation, orthogonality check first, then |cor| grid vs L, row max, mean

In [ ]:
ld <- pca_da$pca$rotation[, 1:3]
stopifnot(is.matrix(ld), is.numeric(ld), dim(ld) == c(17, 3))
cat("ld orthogonal:", round(max(abs(crossprod(ld)[upper.tri(crossprod(ld))])), 6), "\n")

cm <- abs(cor(L, ld))
dimnames(cm) <- list(paste0("F", 1:3), paste0("PC", 1:3))
print(round(cm, 3))
best <- apply(cm, 1, max)
cat("best |cor| per factor:", round(best, 3), "\n")
cat("C3 mean |cor|:", round(mean(best), 3), "| pass >0.7:", mean(best) > 0.7, "\n")

ld orthogonal: 0 
     PC1   PC2   PC3
F1 0.223 0.947 0.076
F2 0.801 0.347 0.723
F3 0.106 0.400 0.800
best |cor| per factor: 0.947 0.801 0.8 
C3 mean |cor|: 0.849 | pass >0.7: TRUE 


ld orthogonal: 0
     PC1   PC2   PC3
F1 0.223 0.947 0.076
F2 0.801 0.347 0.723
F3 0.106 0.400 0.800
best |cor| per factor: 0.947 0.801 0.8
C3 mean |cor|: 0.849 | pass >0.7: TRUE

---

c3 on the bottom-up loadings: 0.849, pass. f1→pc2 0.947, f2→pc1 0.801, f3→pc3 0.800, permuted as expected, f2 bleeds into pc3 at 0.72. replaces the stale 0.731. bank ld

In [ ]:
save_to_drive(list(tab1_p90_sign = tab1, loadings_bu = ld, c3_bu = best,
                   pca_da_var = pca_da$var_explained), tag = "2026-08-21")

saved 4 objects -> /content/drive/MyDrive/thesis/dlnm-pilot/saves_eod_2026-08-21.tar.gz 


saved 4 objects -> /content/drive/MyDrive/thesis/dlnm-pilot/saves_eod_2026-08-21.tar.gz